# Loan Approval — BUILD Phase

Training models using different SageMaker build options:
1. Built-in Algorithm (XGBoost)
2. Built-in Algorithm (Linear Learner)
3. Bring Your Own script (Random Forest via SKLearn container)
4. Autopilot (AutoML)
5. Local mode testing

Using the preprocessed loan approval dataset from the Prepare phase.

In [ ]:
!pip install -q boto3 pandas numpy scikit-learn

In [ ]:
import boto3
import pandas as pd
import numpy as np
import time
import json
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## Setup

In [ ]:
region = boto3.session.Session().region_name
sts = boto3.client('sts')
caller = sts.get_caller_identity()
account_id = caller['Account']
arn = caller['Arn']

if ':assumed-role/' in arn:
    role_name = arn.split(':assumed-role/')[1].split('/')[0]
    role = f"arn:aws:iam::{account_id}:role/{role_name}"
else:
    role = arn

bucket = f"sagemaker-{region}-{account_id}"
prefix = "loan-approval-lab"
s3 = boto3.client('s3')
sm = boto3.client('sagemaker')

print(f"Region: {region}")
print(f"Bucket: {bucket}")

## Prepare training data

Loading raw data and running the same preprocessing from the prepare phase.

In [ ]:
df = pd.read_csv('../data/loan_data.csv')

df = df.drop('customer_id', axis=1)
df['income_per_exp_year'] = df['income'] / (df['employment_years'] + 1)
df['loan_to_income'] = df['loan_amount'] / df['income']
df['risk_score'] = df['late_payments'] * 10 + df['debt_to_income'] * 100

home_dummies = pd.get_dummies(df['home_ownership'], prefix='home').astype(int)
purpose_dummies = pd.get_dummies(df['purpose'], prefix='purpose').astype(int)
df['gender_male'] = (df['gender'] == 'male').astype(int)
df['gender_female'] = (df['gender'] == 'female').astype(int)
df = pd.concat([df, home_dummies, purpose_dummies], axis=1)
df = df.drop(['home_ownership', 'purpose', 'gender'], axis=1)

cols = ['loan_approved'] + [c for c in df.columns if c != 'loan_approved']
df = df[cols]

train, test = train_test_split(df, test_size=0.2, random_state=42, stratify=df['loan_approved'])

X_train = train.iloc[:, 1:]
y_train = train.iloc[:, 0]
X_test = test.iloc[:, 1:]
y_test = test.iloc[:, 0]

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Features: {list(X_train.columns)}")

---
## 1. Local Mode — Quick prototyping

Before submitting expensive cloud training jobs, test locally to make sure the code works.

In [ ]:
# quick local test with logistic regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
lr_preds = lr.predict(X_test)
lr_acc = accuracy_score(y_test, lr_preds)

print(f"Logistic Regression (local) — Accuracy: {lr_acc:.4f}")
print(f"\n{classification_report(y_test, lr_preds)}")

Local mode helps validate that our data pipeline is correct before spending on cloud compute. The logistic regression gives us a baseline to beat.

---
## 2. Built-in Algorithm — XGBoost (local simulation)

SageMaker's built-in XGBoost is one of the most popular algorithms for tabular data. Here we simulate what the managed training job does.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

# simulating XGBoost built-in with gradient boosting
# in production this would be: sagemaker.estimator.Estimator(image_uri=xgboost_image, ...)
xgb = GradientBoostingClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    random_state=42
)
xgb.fit(X_train, y_train)
xgb_preds = xgb.predict(X_test)
xgb_acc = accuracy_score(y_test, xgb_preds)

print(f"XGBoost (built-in sim) — Accuracy: {xgb_acc:.4f}")
print(f"\n{classification_report(y_test, xgb_preds)}")

In [ ]:
# feature importance from xgboost
feat_imp = pd.Series(xgb.feature_importances_, index=X_train.columns)
feat_imp = feat_imp.sort_values(ascending=False)

print("Top 10 features by importance:")
for feat, imp in feat_imp.head(10).items():
    print(f"  {feat:25s} {imp:.4f}")

---
## 3. Built-in Algorithm — Linear Learner (local simulation)

Linear Learner is SageMaker's built-in linear model. Good for understanding which features have linear relationships with the target.

In [ ]:
from sklearn.preprocessing import StandardScaler

# linear models need scaled features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# simulating Linear Learner
ll = LogisticRegression(max_iter=2000, C=1.0, random_state=42)
ll.fit(X_train_scaled, y_train)
ll_preds = ll.predict(X_test_scaled)
ll_acc = accuracy_score(y_test, ll_preds)

print(f"Linear Learner (built-in sim) — Accuracy: {ll_acc:.4f}")
print(f"\n{classification_report(y_test, ll_preds)}")

In [ ]:
# coefficient analysis — which features push toward approval/rejection
coef_df = pd.DataFrame({
    'feature': X_train.columns,
    'coefficient': ll.coef_[0]
}).sort_values('coefficient', ascending=False)

print("Top factors for APPROVAL (positive coef):")
for _, row in coef_df.head(5).iterrows():
    print(f"  {row['feature']:25s} {row['coefficient']:+.4f}")

print("\nTop factors for REJECTION (negative coef):")
for _, row in coef_df.tail(5).iterrows():
    print(f"  {row['feature']:25s} {row['coefficient']:+.4f}")

---
## 4. Bring Your Own — Custom Random Forest script

When built-in algorithms don't fit your needs, you bring your own training script. SageMaker runs it inside a managed container (SKLearn, PyTorch, TensorFlow, etc).

The script is at `scripts/custom_train.py` — it takes hyperparameters as arguments and saves the model.

In [ ]:
# show the custom training script
print(open('../build-phase/scripts/custom_train.py').read())

In [ ]:
# run locally to validate before cloud submission
rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)
rf_acc = accuracy_score(y_test, rf_preds)

print(f"Random Forest (BYO script sim) — Accuracy: {rf_acc:.4f}")
print(f"\n{classification_report(y_test, rf_preds)}")

In [ ]:
# in production, you'd submit this as:
# from sagemaker.sklearn import SKLearn
# estimator = SKLearn(
#     entry_point='custom_train.py',
#     role=role,
#     instance_type='ml.m5.large',
#     framework_version='1.2-1',
#     hyperparameters={'n-estimators': 100, 'max-depth': 5}
# )
# estimator.fit({'train': train_s3_path})
print("BYO workflow: write script → test locally → submit to SageMaker managed infra")

---
## 5. Autopilot — Automated ML

SageMaker Autopilot automatically explores algorithms, feature engineering, and hyperparameters. You just provide data and it finds the best model.

Below we simulate what Autopilot does internally — trying multiple algorithms and selecting the best.

In [ ]:
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# autopilot tries multiple candidates
candidates = {
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=100, max_depth=4, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'SVM': SVC(kernel='rbf', random_state=42)
}

results = []
print("Autopilot candidate evaluation:")
print("-" * 50)

for name, model in candidates.items():
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, preds)
    results.append({'model': name, 'accuracy': acc})
    print(f"  {name:25s} accuracy: {acc:.4f}")

results_df = pd.DataFrame(results).sort_values('accuracy', ascending=False)
best = results_df.iloc[0]
print(f"\nBest model: {best['model']} ({best['accuracy']:.4f})")

In [ ]:
# in production, autopilot is launched with:
# sm.create_auto_ml_job_v2(
#     AutoMLJobName='loan-approval-autopilot',
#     AutoMLJobInputDataConfig=[{
#         'ChannelType': 'training',
#         'DataSource': {'S3DataSource': {'S3Uri': train_s3_uri, 'S3DataType': 'S3Prefix'}}
#     }],
#     OutputDataConfig={'S3OutputPath': output_s3_uri},
#     AutoMLProblemTypeConfig={'TabularJobConfig': {'TargetAttributeName': 'loan_approved'}},
#     RoleArn=role
# )
print("Autopilot: provide data + target column → get best model automatically")

---
## 6. JumpStart — Pre-trained models

SageMaker JumpStart provides pre-trained models that you can fine-tune or deploy directly. For tabular data, JumpStart offers LightGBM, CatBoost, TabTransformer, etc.

Here we simulate fine-tuning a pre-trained approach using a similar boosting model.

In [ ]:
# simulating JumpStart's LightGBM-like model
# in production: sagemaker.jumpstart.estimator.JumpStartEstimator(model_id='lightgbm-classification-model')

jumpstart_model = GradientBoostingClassifier(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    random_state=42
)
jumpstart_model.fit(X_train, y_train)
js_preds = jumpstart_model.predict(X_test)
js_acc = accuracy_score(y_test, js_preds)

print(f"JumpStart LightGBM (sim) — Accuracy: {js_acc:.4f}")
print(f"\n{classification_report(y_test, js_preds)}")

---
## Model Comparison Summary

In [ ]:
summary = pd.DataFrame([
    {'Method': 'Local Mode (Logistic Regression)', 'Accuracy': lr_acc, 'SageMaker Feature': 'Local Mode'},
    {'Method': 'Built-in XGBoost', 'Accuracy': xgb_acc, 'SageMaker Feature': 'Built-in Algorithms'},
    {'Method': 'Built-in Linear Learner', 'Accuracy': ll_acc, 'SageMaker Feature': 'Built-in Algorithms'},
    {'Method': 'BYO Random Forest', 'Accuracy': rf_acc, 'SageMaker Feature': 'Bring Your Own'},
    {'Method': 'Autopilot Best', 'Accuracy': best['accuracy'], 'SageMaker Feature': 'Autopilot'},
    {'Method': 'JumpStart LightGBM', 'Accuracy': js_acc, 'SageMaker Feature': 'JumpStart'},
]).sort_values('Accuracy', ascending=False)

print("Model Comparison:")
print("=" * 70)
for _, row in summary.iterrows():
    print(f"  {row['Method']:35s} {row['Accuracy']:.4f}   ({row['SageMaker Feature']})")
print("=" * 70)
print(f"\nWinner: {summary.iloc[0]['Method']}")

## Cloud training job example (reference)

Showing how the XGBoost training would be submitted as a real SageMaker training job using Boto3.

In [ ]:
# reference code — how to submit actual SageMaker training job
# not executing to avoid costs

training_job_config = {
    'TrainingJobName': f'loan-xgboost-{int(time.time())}',
    'AlgorithmSpecification': {
        'TrainingImage': f'683313688378.dkr.ecr.{region}.amazonaws.com/sagemaker-xgboost:1.7-1',
        'TrainingInputMode': 'File'
    },
    'RoleArn': role,
    'InputDataConfig': [{
        'ChannelName': 'train',
        'DataSource': {
            'S3DataSource': {
                'S3DataType': 'S3Prefix',
                'S3Uri': f's3://{bucket}/{prefix}/processed/train/',
                'S3DataDistributionType': 'FullyReplicated'
            }
        },
        'ContentType': 'text/csv'
    }],
    'OutputDataConfig': {
        'S3OutputPath': f's3://{bucket}/{prefix}/models/'
    },
    'ResourceConfig': {
        'InstanceType': 'ml.m5.large',
        'InstanceCount': 1,
        'VolumeSizeInGB': 10
    },
    'HyperParameters': {
        'objective': 'binary:logistic',
        'num_round': '100',
        'max_depth': '4',
        'eta': '0.1'
    },
    'StoppingCondition': {'MaxRuntimeInSeconds': 600}
}

print("Training job config (ready to submit):")
print(json.dumps(training_job_config, indent=2, default=str))
print("\n# To actually run: sm.create_training_job(**training_job_config)")

## Takeaways

- **Local Mode**: Fast iteration, zero cost — use for debugging
- **Built-in Algorithms**: No code needed, auto-scaling, optimized containers
- **JumpStart**: Pre-trained models, one-click fine-tuning
- **Autopilot**: Full AutoML — best for quick baselines
- **BYO Script**: Maximum flexibility when you need custom logic
- **Cloud Training**: Submit to managed infra when local tests pass